# LeetCode 325 — Maximum Size Subarray Sum Equals k bằng Python

Link đề: https://leetcode.com/problems/maximum-size-subarray-sum-equals-k/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def maxSubArrayLen(self, nums: List[int], k: int) -> int:
```

> **Lưu ý:** đây là bài **Premium**. Đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **độ dài lớn nhất** của một mảng con **liên tiếp** có tổng đúng bằng `k` (không có thì trả `0`); mảng có thể chứa **số âm**.
2. Biết biến đổi "tổng đoạn" thành **hiệu hai tổng tiền tố**: `sum(nums[j+1..i]) = P[i] - P[j]`.
3. Đi từ **vét cạn ba vòng lặp** `O(n³)` → **hai vòng lặp với tổng chạy** `O(n²)` → **bảng băm lưu chỉ số xuất hiện đầu tiên của mỗi tổng tiền tố** `O(n)` → **sắp xếp + tìm kiếm nhị phân** (không dùng băm) `O(n log n)`.
4. Hiểu **vì sao chỉ lưu lần xuất hiện ĐẦU TIÊN** của mỗi tổng tiền tố (để đoạn dài nhất) và **vì sao phải khởi tạo `{0: -1}`**.
5. Hiểu **vì sao cửa sổ trượt hai con trỏ KHÔNG dùng được** khi có số âm.
6. Tránh các lỗi kinh điển và nhận ra họ hàng của bài (560, 525, 209, 523).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`, trả về **độ dài lớn nhất của một mảng con** có tổng bằng `k`. Nếu không có mảng con nào như vậy, trả về `0`.

**Ví dụ 1**

- Input: `nums = [1,-1,5,-2,3], k = 3`
- Output: `4`
- Giải thích: mảng con `[1, -1, 5, -2]` có tổng `3` và dài nhất.

**Ví dụ 2**

- Input: `nums = [-2,-1,2,1], k = 1`
- Output: `2`
- Giải thích: mảng con `[-1, 2]` có tổng `1` và dài nhất.

**Ràng buộc:** `1 <= nums.length <= 2·10^5`; `-10^4 <= nums[i] <= 10^4`; `-10^9 <= k <= 10^9`.

In [1]:
from typing import List
from itertools import accumulate
from bisect import bisect_left
import random
import time

examples = [
    (([1, -1, 5, -2, 3], 3), 4),
    (([-2, -1, 2, 1], 1), 2),
]
for (nums, k), expected in examples:
    print("nums =", nums, "| k =", k, "| kỳ vọng =", expected)

nums = [1, -1, 5, -2, 3] | k = 3 | kỳ vọng = 4
nums = [-2, -1, 2, 1] | k = 1 | kỳ vọng = 2


## 3. Kiến thức nền

### 3.1. Mảng con (subarray) và tổng tiền tố

Mảng con là một đoạn **liên tiếp** `nums[j..i]`. Gọi `P[0] = 0`, `P[t] = nums[0] + ... + nums[t-1]` (tổng `t` phần tử đầu). Khi đó

`sum(nums[j..i]) = P[i+1] - P[j]`.

Muốn tổng đoạn bằng `k` ta cần hai chỉ số tiền tố `a < b` với `P[b] - P[a] = k`, độ dài đoạn là `b - a`.
#### Code demo

In [2]:
nums = [1, -1, 5, -2, 3]
P = [0] + list(accumulate(nums))
print("P =", P)
print("tổng nums[0..3] =", sum(nums[0:4]), "= P[4] - P[0] =", P[4] - P[0])

P = [0, 1, 0, 5, 3, 6]
tổng nums[0..3] = 3 = P[4] - P[0] = 3


### 3.2. Bảng băm (dict) và tra cứu `O(1)`

`dict` cho phép hỏi "khóa này đã có chưa, giá trị là gì" trong `O(1)` trung bình. Ta sẽ dùng nó để hỏi: "tổng tiền tố `P[b] - k` đã từng xuất hiện chưa, **lần đầu ở đâu**?"
#### Code demo

In [3]:
first = {}
for idx, value in enumerate([5, 7, 5, 9, 7]):
    if value not in first:          # chỉ ghi lần xuất hiện ĐẦU TIÊN
        first[value] = idx
print(first)                        # {5: 0, 7: 1, 9: 3}
print(first.get(7), first.get(100, "không có"))

{5: 0, 7: 1, 9: 3}
1 không có


### 3.3. Số âm làm hỏng tính đơn điệu

Nếu mọi số `≥ 0`, tổng tiền tố **không giảm** nên có thể dùng hai con trỏ (mở rộng phải, co trái). Khi có số âm, tổng tiền tố **lên xuống** và thêm một phần tử có thể làm tổng giảm: không còn quy tắc "tổng lớn quá thì co trái". Đó là lý do bài này dùng **bảng băm**, không phải cửa sổ trượt.
#### Code demo

In [4]:
nums = [1, -1, 5, -2, 3]
print("tổng tiền tố:", [0] + list(accumulate(nums)), "-> không đơn điệu")

tổng tiền tố: [0, 1, 0, 5, 3, 6] -> không đơn điệu


### 3.4. Tìm kiếm nhị phân với `bisect_left`

`bisect_left(a, x)` trả vị trí chèn `x` vào mảng đã sắp `a` sao cho `a[:pos] < x`. Nếu `a[pos] == x` thì `x` có trong `a`. Ta dùng nó ở Cách 4 để tra cứu mà không cần băm.
#### Code demo

In [5]:
a = [2, 4, 4, 7]
pos = bisect_left(a, 4)
print(pos, a[pos] == 4)                       # 1 True
pos = bisect_left(a, 5)
print(pos, pos < len(a) and a[pos] == 5)      # 3 False

1 True
3 False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`nums = [1, -1, 5, -2, 3]`, `k = 3`. Tổng tiền tố `P = [0, 1, 0, 5, 3, 6]`.

Với mỗi vị trí kết thúc `b` (chỉ số tiền tố), ta cần một `a < b` có `P[a] = P[b] - 3`:

| `b` | `P[b]` | cần `P[a] =` | `a` xa nhất (nhỏ nhất) | độ dài `b-a` |
|---|---|---|---|---|
| 1 | 1 | -2 | không có | — |
| 2 | 0 | -3 | không có | — |
| 3 | 5 | 2 | không có | — |
| 4 | 3 | 0 | `a = 0` (cũng có `a = 2`) | **4** |
| 5 | 6 | 3 | `a = 4` | 1 |

Cặp `(a, b) = (0, 4)` cho đoạn `nums[0..3]` dài `4`. Chú ý: giá trị `0` xuất hiện ở `a = 0` **và** `a = 2`; để đoạn **dài nhất** ta chọn `a` **nhỏ nhất**, tức **lần xuất hiện đầu tiên**.

## 5. Cách 1 — Brute Force / baseline: ba vòng lặp

**Ý tưởng.** Thử mọi cặp `(j, i)` với `j ≤ i`, tính tổng `nums[j..i]` bằng một vòng lặp thứ ba, nếu bằng `k` thì cập nhật độ dài lớn nhất.
#### Code demo

In [6]:
def max_len_bruteforce(nums, k):
    n = len(nums)
    best = 0
    for j in range(n):
        for i in range(j, n):
            total = 0
            for t in range(j, i + 1):
                total += nums[t]
            if total == k:
                best = max(best, i - j + 1)
    return best


for (args, expected) in examples:
    print(args, "->", max_len_bruteforce(*args), "| kỳ vọng:", expected)

([1, -1, 5, -2, 3], 3) -> 4 | kỳ vọng: 4
([-2, -1, 2, 1], 1) -> 2 | kỳ vọng: 2


**Độ phức tạp (Cách 1).** **Time `O(n³)`**, **Space `O(1)`**. Với `n = 2·10^5` hoàn toàn không khả thi.

**Nút thắt.** Tổng đoạn bị tính lại từ đầu ở mỗi cặp `(j, i)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — tổng chạy.** Cố định điểm đầu `j`, kéo điểm cuối `i` sang phải và cộng dồn: `O(n²)`.

**Bước 2 — hiệu hai tổng tiền tố.** `sum(j..i) = P[i+1] - P[j]`. Duyệt `i` từ trái sang phải, giữ tổng tiền tố hiện tại `s`. Ta cần tìm **tiền tố cũ** `P[a] = s - k` với `a` **nhỏ nhất** (đoạn dài nhất).

**Bước 3 — bảng băm lưu lần xuất hiện đầu tiên.** `first[v]` = chỉ số nhỏ nhất `a` có `P[a] = v`. Khi duyệt tới `i`, tra `first.get(s - k)`; nếu có thì độ dài ứng viên là `i - first[s - k]`. Chỉ ghi `first[s]` **nếu chưa có** (để giữ lần đầu). Khởi tạo `first[0] = -1` để bao trường hợp đoạn bắt đầu từ đầu mảng.

Tổng thời gian `O(n)`.

## 7. Các cách tiếp theo

### Cách 2 — Hai vòng lặp với tổng chạy `O(n²)`

**Giải thích.** Cố định `j`, cộng dồn `total` khi `i` đi sang phải; mỗi lần `total == k` thì cập nhật.
#### Code demo

In [7]:
def max_len_quadratic(nums, k):
    n = len(nums)
    best = 0
    for j in range(n):
        total = 0
        for i in range(j, n):
            total += nums[i]
            if total == k and i - j + 1 > best:
                best = i - j + 1
    return best


for (args, expected) in examples:
    print(args, "->", max_len_quadratic(*args), "| kỳ vọng:", expected)

([1, -1, 5, -2, 3], 3) -> 4 | kỳ vọng: 4
([-2, -1, 2, 1], 1) -> 2 | kỳ vọng: 2


**Độ phức tạp Cách 2:** **Time `O(n²)`** (cỡ `2·10^10` phép với `n = 2·10^5`, quá chậm), **Space `O(1)`**.

### Cách 3 — Tổng tiền tố + bảng băm (cách nộp chuẩn)

**Giải thích.** Như mục 6: duyệt một lượt, lưu lần xuất hiện đầu tiên của mỗi tổng tiền tố.
#### Code demo

In [8]:
def max_len_hash(nums, k):
    first = {0: -1}                       # tổng tiền tố rỗng xuất hiện "trước chỉ số 0"
    s = 0
    best = 0
    for i, x in enumerate(nums):
        s += x
        if s - k in first:
            best = max(best, i - first[s - k])
        if s not in first:                # chỉ giữ lần xuất hiện ĐẦU TIÊN
            first[s] = i
    return best


for (args, expected) in examples:
    print(args, "->", max_len_hash(*args), "| kỳ vọng:", expected)

([1, -1, 5, -2, 3], 3) -> 4 | kỳ vọng: 4
([-2, -1, 2, 1], 1) -> 2 | kỳ vọng: 2


**Độ phức tạp Cách 3:** **Time `O(n)`** trung bình, **Space `O(n)`** (tối đa `n + 1` khóa).

### Cách 4 — Sắp xếp các cặp (tổng tiền tố, chỉ số) + tìm kiếm nhị phân (không dùng băm)

**Giải thích.**

1. Tính `P[0..n]`.
2. Tạo danh sách cặp `(P[t], t)` rồi **sắp xếp**; với các tổng bằng nhau, chỉ số nhỏ nhất đứng trước. Giữ lại **phần tử đầu của mỗi nhóm giá trị bằng nhau**: hai mảng song song `keys` (giá trị phân biệt tăng dần) và `idxs` (chỉ số xuất hiện đầu tiên).
3. Với mỗi `b = 1..n`: tìm `P[b] - k` trong `keys` bằng `bisect_left`; nếu thấy tại vị trí `p` với `idxs[p] < b` thì ứng viên `b - idxs[p]`.

Thay vì dùng `dict`, ta dùng sắp xếp + nhị phân — hữu ích khi muốn tránh va chạm băm hoặc dùng ngôn ngữ không có bảng băm tiện.
#### Code demo

In [9]:
def max_len_sorted(nums, k):
    n = len(nums)
    P = [0] * (n + 1)
    for t in range(n):
        P[t + 1] = P[t] + nums[t]
    pairs = sorted((P[t], t) for t in range(n + 1))
    keys, idxs = [], []
    for value, t in pairs:
        if not keys or keys[-1] != value:   # phần tử đầu của nhóm = chỉ số nhỏ nhất
            keys.append(value)
            idxs.append(t)
    best = 0
    for b in range(1, n + 1):
        target = P[b] - k
        p = bisect_left(keys, target)
        if p < len(keys) and keys[p] == target and idxs[p] < b:
            best = max(best, b - idxs[p])
    return best


for (args, expected) in examples:
    print(args, "->", max_len_sorted(*args), "| kỳ vọng:", expected)

([1, -1, 5, -2, 3], 3) -> 4 | kỳ vọng: 4
([-2, -1, 2, 1], 1) -> 2 | kỳ vọng: 2


**Độ phức tạp Cách 4:** **Time `O(n log n)`** (sắp xếp + `n` lần nhị phân), **Space `O(n)`**.

> Cách 3 là bản nộp thông dụng. Cách 4 cho thấy vai trò thực chất của bảng băm: "tra cứu chỉ số đầu tiên của một giá trị".

## 8. TRACE từng bước

Ta trace Cách 3 trên `nums = [1, -1, 5, -2, 3]`, `k = 3`, in `i`, `x`, tổng tiền tố `s`, `s - k`, bảng `first` và đáp án hiện tại.
#### Code demo

In [10]:
def trace_hash(nums, k):
    first = {0: -1}
    s = 0
    best = 0
    print(f"nums={nums}, k={k}, first ban đầu={first}")
    for i, x in enumerate(nums):
        s += x
        note = []
        if s - k in first:
            length = i - first[s - k]
            note.append(f"tìm thấy s-k={s - k} tại {first[s - k]} -> độ dài {length}")
            best = max(best, length)
        else:
            note.append(f"s-k={s - k} chưa có")
        if s not in first:
            first[s] = i
            note.append(f"ghi first[{s}]={i}")
        else:
            note.append(f"first[{s}]={first[s]} đã có, giữ nguyên")
        print(f"i={i} x={x:>2} s={s:>2} | {'; '.join(note)} | best={best}")
    print("first cuối:", first, "=> đáp án", best)
    return best


trace_hash([1, -1, 5, -2, 3], 3)
print()
trace_hash([-2, -1, 2, 1], 1)

nums=[1, -1, 5, -2, 3], k=3, first ban đầu={0: -1}
i=0 x= 1 s= 1 | s-k=-2 chưa có; ghi first[1]=0 | best=0
i=1 x=-1 s= 0 | s-k=-3 chưa có; first[0]=-1 đã có, giữ nguyên | best=0
i=2 x= 5 s= 5 | s-k=2 chưa có; ghi first[5]=2 | best=0
i=3 x=-2 s= 3 | tìm thấy s-k=0 tại -1 -> độ dài 4; ghi first[3]=3 | best=4
i=4 x= 3 s= 6 | tìm thấy s-k=3 tại 3 -> độ dài 1; ghi first[6]=4 | best=4
first cuối: {0: -1, 1: 0, 5: 2, 3: 3, 6: 4} => đáp án 4

nums=[-2, -1, 2, 1], k=1, first ban đầu={0: -1}
i=0 x=-2 s=-2 | s-k=-3 chưa có; ghi first[-2]=0 | best=0
i=1 x=-1 s=-3 | s-k=-4 chưa có; ghi first[-3]=1 | best=0
i=2 x= 2 s=-1 | tìm thấy s-k=-2 tại 0 -> độ dài 2; ghi first[-1]=2 | best=2
i=3 x= 1 s= 0 | tìm thấy s-k=-1 tại 2 -> độ dài 1; first[0]=-1 đã có, giữ nguyên | best=2
first cuối: {0: -1, -2: 0, -3: 1, -1: 2} => đáp án 2


2

Bảng tóm tắt (`[1, -1, 5, -2, 3]`, `k = 3`):

| `i` | `x` | `s` | `s - k` | thấy trong `first`? | `best` |
|---|---|---|---|---|---|
| 0 | 1 | 1 | -2 | không | 0 |
| 1 | -1 | 0 | -3 | không (nhưng `first[0] = -1` đã có, giữ) | 0 |
| 2 | 5 | 5 | 2 | không | 0 |
| 3 | -2 | 3 | 0 | có tại `-1` → độ dài `3 - (-1) = 4` | **4** |
| 4 | 3 | 6 | 3 | có tại `3` → độ dài `1` | 4 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3. Đúng `class Solution`, đúng `maxSubArrayLen(self, nums: List[int], k: int) -> int`.
#### Code demo

In [11]:
class Solution:
    def maxSubArrayLen(self, nums: List[int], k: int) -> int:
        first = {0: -1}                    # tổng tiền tố -> chỉ số xuất hiện ĐẦU TIÊN
        s = 0
        best = 0
        for i, x in enumerate(nums):
            s += x
            if s - k in first:
                best = max(best, i - first[s - k])
            if s not in first:
                first[s] = i
        return best


sol = Solution()
for (args, expected) in examples:
    assert sol.maxSubArrayLen(*args) == expected, (args, expected)
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Tương đương:** mảng con `nums[a..b-1]` có tổng `k` ⟺ `P[b] - P[a] = k` với `0 ≤ a < b ≤ n`. Mọi đáp án là một cặp `(a, b)`.
- **Tối ưu cho từng `b`:** với `b` cố định, độ dài `b - a` lớn nhất khi `a` **nhỏ nhất** thỏa `P[a] = P[b] - k`. `first[P[b] - k]` chính là chỉ số nhỏ nhất đó **trong số các `a < b`** (vì tại thời điểm xử lý `b` bảng `first` chỉ chứa các `P[a]` với `a < b`; ta thêm `P[b]` **sau** khi tra cứu).
- **Chỉ ghi lần đầu:** nếu ghi đè bằng lần sau thì `a` lớn hơn ⇒ đoạn ngắn hơn ⇒ có thể bỏ lỡ đáp án tối ưu.
- **Khởi tạo `{0: -1}`:** đại diện `P[0] = 0` (tiền tố rỗng) — cần cho các đoạn bắt đầu từ chỉ số `0`. (Trong mã dùng chỉ số kết thúc `i = b - 1` nên `a = 0` tương ứng khóa `-1`.)
- **Không cần giả sử về dấu:** thuật toán chỉ dựa vào đẳng thức hiệu, nên đúng với số âm, số `0`, mọi `k`.
- **Cách 4:** cùng ý tưởng; `idxs[p]` là chỉ số đầu tiên của giá trị `P[b] - k`; điều kiện `idxs[p] < b` loại trường hợp `k = 0` khi chính `b` là lần đầu (đoạn rỗng).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (≤ `2·10^5`).

| Cách | Time | Space |
|---|---|---|
| 1. Ba vòng lặp | `O(n³)` | `O(1)` |
| 2. Hai vòng lặp, tổng chạy | `O(n²)` | `O(1)` |
| 3. Tiền tố + dict | `O(n)` | `O(n)` |
| 4. Sắp xếp + nhị phân | `O(n log n)` | `O(n)` |

Với `n = 2·10^5`: Cách 2 cỡ `2·10^10` phép (quá chậm), Cách 3 cỡ `2·10^5` phép, Cách 4 cỡ `3.5·10^6` phép. Giá trị tiền tố tối đa cỡ `2·10^9` — số nguyên Python không tràn.

## 12. Bộ test

`oracle` là brute force ba vòng lặp (`max_len_bruteforce`), **độc lập** với cách dùng tổng tiền tố. Ta kiểm tra ví dụ của đề, biên, tính chất và ngẫu nhiên có seed.
#### Code demo

In [12]:
def oracle(nums, k):
    return max_len_bruteforce(nums, k)


impls = {
    "bruteforce": max_len_bruteforce,
    "quadratic": max_len_quadratic,
    "hash": max_len_hash,
    "sorted": max_len_sorted,
    "submit": sol.maxSubArrayLen,
}

for (args, expected) in examples:
    for key, fn in impls.items():
        assert fn(*args) == expected, (key, args, expected, fn(*args))
print("PASS ví dụ của đề")

PASS ví dụ của đề


### 12.1. Các trường hợp biên
#### Code demo

In [13]:
edge_cases = [
    (([5], 5), 1, "một phần tử bằng k"),
    (([5], 3), 0, "một phần tử khác k"),
    (([0], 0), 1, "phần tử 0 và k = 0"),
    (([0, 0, 0, 0], 0), 4, "toàn số 0, k = 0"),
    (([1, 2, 3], 6), 3, "cả mảng"),
    (([1, 2, 3], 100), 0, "k không đạt được"),
    (([-1, -1, -1], -2), 2, "toàn số âm, k âm"),
    (([-1, -1, -1], 2), 0, "k dương không đạt được với số âm"),
    (([1, -1, 1, -1], 0), 4, "tổng tiền tố lặp lại, k = 0 (cả mảng)"),
    (([3, 1, -4, 2], 0), 3, "đoạn giữa/đầu tổng 0: [3,1,-4]"),
    (([1, 2, 3, 4], 3), 2, "hai đoạn tổng 3: [3] và [1,2]; chọn dài hơn"),
    (([2, 2, 2, 2], 4), 2, "nhiều đoạn dài bằng nhau"),
    (([10 ** 4] * 5, 5 * 10 ** 4), 5, "giá trị biên 10^4"),
    (([-10 ** 4] * 5, -3 * 10 ** 4), 3, "giá trị biên âm"),
]
for args, expected, label in edge_cases:
    for key, fn in impls.items():
        assert fn(*args) == expected, (key, label, args, expected, fn(*args))
    print("PASS", label)

PASS một phần tử bằng k
PASS một phần tử khác k
PASS phần tử 0 và k = 0
PASS toàn số 0, k = 0
PASS cả mảng
PASS k không đạt được
PASS toàn số âm, k âm
PASS k dương không đạt được với số âm
PASS tổng tiền tố lặp lại, k = 0 (cả mảng)
PASS đoạn giữa/đầu tổng 0: [3,1,-4]
PASS hai đoạn tổng 3: [3] và [1,2]; chọn dài hơn
PASS nhiều đoạn dài bằng nhau
PASS giá trị biên 10^4
PASS giá trị biên âm


### 12.2. Tính chất và phản ví dụ

- Kết quả luôn `0 ≤ ans ≤ n`.
- **Đảo mảng** không đổi đáp án (đoạn liên tiếp vẫn liên tiếp).
- **Đổi dấu** mọi phần tử và `k` không đổi đáp án.
- Nếu `ans > 0` thì tồn tại một đoạn đúng độ dài đó có tổng `k` (kiểm tra bằng cách quét).
- **Phản ví dụ** cho cửa sổ trượt hai con trỏ khi có số âm: `[1, -1, 5, -2, 3]`, `k = 3`.
#### Code demo

In [14]:
rng = random.Random(325)
for _ in range(500):
    n = rng.randint(1, 12)
    nums = [rng.randint(-4, 4) for _ in range(n)]
    k = rng.randint(-6, 6)
    ans = max_len_hash(nums, k)
    assert 0 <= ans <= n
    assert max_len_hash(nums[::-1], k) == ans
    assert max_len_hash([-x for x in nums], -k) == ans
    if ans > 0:
        assert any(sum(nums[j:j + ans]) == k for j in range(n - ans + 1))
print("PASS: giới hạn [0, n], đảo mảng, đổi dấu, tồn tại đoạn đúng độ dài")


def sai_hai_con_tro(nums, k):
    """SAI khi có số âm: co cửa sổ khi tổng vượt quá k."""
    left, total, best = 0, 0, 0
    for right, x in enumerate(nums):
        total += x
        while total > k and left <= right:
            total -= nums[left]
            left += 1
        if total == k:
            best = max(best, right - left + 1)
    return best


bad = ([1, -1, 5, -2, 3], 3)
print("hai con trỏ:", sai_hai_con_tro(*bad), "| đúng:", max_len_hash(*bad))
assert sai_hai_con_tro(*bad) != max_len_hash(*bad)

PASS: giới hạn [0, n], đảo mảng, đổi dấu, tồn tại đoạn đúng độ dài
hai con trỏ: 0 | đúng: 4


### 12.3. Kiểm tra ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [15]:
rng = random.Random(2325)
for trial in range(4000):
    n = rng.randint(1, 12)
    lo, hi = rng.choice([(-3, 3), (0, 3), (-1, 1), (-10, 10), (-10 ** 4, 10 ** 4)])
    nums = [rng.randint(lo, hi) for _ in range(n)]
    if rng.random() < 0.5:
        i = rng.randint(0, n - 1)
        j = rng.randint(i, n - 1)
        k = sum(nums[i:j + 1])                    # k chắc chắn đạt được
    else:
        k = rng.randint(-8, 8)
    expected = oracle(nums, k)
    for key, fn in impls.items():
        assert fn(nums, k) == expected, (key, nums, k, expected, fn(nums, k))
print("PASS 4000 test ngẫu nhiên với oracle ba vòng lặp")

# So sánh các cách nhanh với O(n^2) ở n trung bình
for _ in range(20):
    n = 1500
    nums = [rng.randint(-5, 5) for _ in range(n)]
    k = rng.randint(-10, 10)
    exp = max_len_quadratic(nums, k)
    assert max_len_hash(nums, k) == exp and max_len_sorted(nums, k) == exp and sol.maxSubArrayLen(nums, k) == exp
print("PASS 20 test n = 1500 (so với O(n^2))")

# n = 2 * 10^5
N = 200_000
big_cases = [
    ("ngẫu nhiên", [rng.randint(-10 ** 4, 10 ** 4) for _ in range(N)], rng.randint(-10 ** 9, 10 ** 9)),
    ("toàn số 0, k = 0 (đáp án n)", [0] * N, 0),
    ("toàn số 1, k = N", [1] * N, N),
    ("xen kẽ +1/-1, k = 0", [1, -1] * (N // 2), 0),
    ("nhỏ ngẫu nhiên -1..1, k = 0", [rng.randint(-1, 1) for _ in range(N)], 0),
]
for label, nums, k in big_cases:
    row = []
    answers = set()
    for key in ("hash", "sorted", "submit"):
        t0 = time.perf_counter()
        r = impls[key](nums, k)
        dt = (time.perf_counter() - t0) * 1000
        answers.add(r)
        row.append(f"{key}={dt:7.1f}ms")
    assert len(answers) == 1, (label, answers)
    print(f"{label:<30} | ans={answers.pop():>7} | " + " ".join(row))
print("PASS n = 2*10^5")

PASS 4000 test ngẫu nhiên với oracle ba vòng lặp


PASS 20 test n = 1500 (so với O(n^2))


ngẫu nhiên                     | ans=      0 | hash=   65.0ms sorted=  198.3ms submit=   44.9ms
toàn số 0, k = 0 (đáp án n)    | ans= 200000 | hash=   18.6ms sorted=   67.6ms submit=   18.8ms


toàn số 1, k = N               | ans= 200000 | hash=   24.5ms sorted=   85.4ms submit=   26.8ms
xen kẽ +1/-1, k = 0            | ans= 200000 | hash=   23.8ms sorted=   97.4ms submit=   20.8ms


nhỏ ngẫu nhiên -1..1, k = 0    | ans= 123649 | hash=   30.6ms sorted=  134.5ms submit=   22.8ms
PASS n = 2*10^5


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên khởi tạo `{0: -1}`
#### Code sai

In [16]:
def sai_quen_khoi_tao(nums, k):
    first = {}                              # SAI: thiếu 0 -> -1
    s = best = 0
    for i, x in enumerate(nums):
        s += x
        if s - k in first:
            best = max(best, i - first[s - k])
        if s not in first:
            first[s] = i
    return best


args = ([1, 2], 3)
print("sai:", sai_quen_khoi_tao(*args), "| đúng:", sol.maxSubArrayLen(*args))
assert sai_quen_khoi_tao(*args) != sol.maxSubArrayLen(*args)

sai: 0 | đúng: 2


**Vì sao sai:** đoạn bắt đầu từ chỉ số `0` có tổng tiền tố đúng bằng `k`, tức `s - k = 0`, cần có khóa `0` (tiền tố rỗng, ở vị trí `-1`). **Code đúng:** `first = {0: -1}`.

### Lỗi 2: Ghi đè chỉ số bằng lần xuất hiện sau
#### Code sai

In [17]:
def sai_ghi_de(nums, k):
    first = {0: -1}
    s = best = 0
    for i, x in enumerate(nums):
        s += x
        if s - k in first:
            best = max(best, i - first[s - k])
        first[s] = i                        # SAI: luôn ghi đè -> mất lần đầu
    return best


args = ([0, 0, 0], 0)
print("sai:", sai_ghi_de(*args), "| đúng:", sol.maxSubArrayLen(*args))
assert sai_ghi_de(*args) != sol.maxSubArrayLen(*args)

sai: 1 | đúng: 3


**Vì sao sai:** muốn đoạn **dài nhất** cần chỉ số **nhỏ nhất** cho mỗi tổng; ghi đè đẩy chỉ số về phía phải nên đoạn ngắn lại. **Code đúng:** `if s not in first: first[s] = i`.

### Lỗi 3: Dùng cửa sổ trượt hai con trỏ khi có số âm
Xem `sai_hai_con_tro` ở mục 12.2: `[1, -1, 5, -2, 3]`, `k = 3` cho kết quả khác đáp án đúng `4`. **Vì sao sai:** thêm số âm làm tổng giảm, "co trái khi tổng quá lớn" có thể bỏ lỡ đoạn đúng. **Code đúng:** bảng băm tổng tiền tố.

### Lỗi 4: Chỉ xét đoạn bắt đầu từ đầu mảng
#### Code sai

In [18]:
def sai_chi_tien_to(nums, k):
    s = best = 0
    for i, x in enumerate(nums):
        s += x
        if s == k:                          # SAI: chỉ xét đoạn nums[0..i]
            best = i + 1
    return best


args = ([-2, -1, 2, 1], 1)
print("sai:", sai_chi_tien_to(*args), "| đúng:", sol.maxSubArrayLen(*args))
assert sai_chi_tien_to(*args) != sol.maxSubArrayLen(*args)

sai: 0 | đúng: 2


**Vì sao sai:** đoạn tối ưu `[-1, 2]` không bắt đầu từ chỉ số `0`. **Code đúng:** so sánh với **mọi** tiền tố cũ qua bảng băm.

### Lỗi 5: Nhầm với bài đếm số mảng con (LeetCode 560)
#### Code sai

In [19]:
def sai_dem_thay_vi_do_dai(nums, k):
    from collections import defaultdict
    cnt = defaultdict(int)
    cnt[0] = 1
    s = res = 0
    for x in nums:
        s += x
        res += cnt[s - k]                   # SAI: đây là ĐẾM số đoạn, không phải độ dài lớn nhất
        cnt[s] += 1
    return res


args = ([1, -1, 5, -2, 3], 3)
print("sai (đếm):", sai_dem_thay_vi_do_dai(*args), "| đúng (độ dài):", sol.maxSubArrayLen(*args))
assert sai_dem_thay_vi_do_dai(*args) != sol.maxSubArrayLen(*args)

sai (đếm): 3 | đúng (độ dài): 4


**Vì sao sai:** bài 560 đếm số đoạn nên lưu **số lần** xuất hiện; bài này cần **độ dài lớn nhất** nên lưu **chỉ số đầu tiên**.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**mảng con liên tiếp có tổng bằng `k`**" và có thể có **số âm** → **tổng tiền tố + bảng băm**.
- "Dài nhất" → lưu **chỉ số đầu tiên** của mỗi tổng; "đếm số đoạn" → lưu **số lần**; "ngắn nhất, số dương" → **hai con trỏ**.
- Luôn khởi tạo `{0: -1}` (tiền tố rỗng).
- Tra cứu **trước**, ghi `s` **sau** (tránh ghép tiền tố với chính nó khi `k = 0`).
- Họ hàng: **560** (đếm mảng con tổng `k`), **525** (mảng con 0/1 cân bằng, đổi `0 → -1`), **523** (tổng chia hết cho `k`, dùng dư), **209** (độ dài nhỏ nhất, số dương, hai con trỏ), **1124** (khoảng dài nhất mà số ngày mệt nhiều hơn).
- Thử: `[0,0,0]` với `k = 0`, `[1,2]` với `k = 3`, mảng toàn âm.

## 15. Biến thể của bài

1. **Đếm số mảng con có tổng `k`** (LeetCode 560): lưu số lần xuất hiện.
2. **Mảng con liên tiếp 0/1 dài nhất có số 0 bằng số 1** (525): đổi `0 → -1`, `k = 0`.
3. **Độ dài nhỏ nhất của mảng con có tổng `≥ target`, số dương** (209): hai con trỏ.
4. **Mảng con tổng chia hết cho `k`** (523): lưu dư tổng tiền tố.
5. **Trả luôn chỉ số `(l, r)` của đoạn tối ưu** thay vì độ dài.
#### Code demo

In [20]:
from collections import defaultdict


def count_subarrays_sum_k(nums, k):                    # 560
    cnt = defaultdict(int)
    cnt[0] = 1
    s = res = 0
    for x in nums:
        s += x
        res += cnt[s - k]
        cnt[s] += 1
    return res


def longest_balanced_01(nums):                         # 525
    return max_len_hash([1 if x == 1 else -1 for x in nums], 0)


def min_len_at_least(target, nums):                    # 209 (số dương)
    left = total = 0
    best = float("inf")
    for right, x in enumerate(nums):
        total += x
        while total >= target:
            best = min(best, right - left + 1)
            total -= nums[left]
            left += 1
    return 0 if best == float("inf") else best


def best_segment(nums, k):
    first = {0: -1}
    s = 0
    best = (0, -1, -1)                                  # (độ dài, l, r)
    for i, x in enumerate(nums):
        s += x
        if s - k in first and i - first[s - k] > best[0]:
            best = (i - first[s - k], first[s - k] + 1, i)
        if s not in first:
            first[s] = i
    return best


print(count_subarrays_sum_k([1, 1, 1], 2), longest_balanced_01([0, 1, 0]), min_len_at_least(7, [2, 3, 1, 2, 4, 3]))
print(best_segment([1, -1, 5, -2, 3], 3))               # (4, 0, 3)

2 2 2
(4, 0, 3)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ba vòng lặp | thử mọi đoạn, cộng lại | `O(n³)` | `O(1)` | oracle `n ≤ 12` |
| 2. Tổng chạy | cố định đầu, cộng dồn cuối | `O(n²)` | `O(1)` | `n` nhỏ, kiểm chứng |
| 3. Tiền tố + dict | `first[s]` chỉ số đầu | `O(n)` | `O(n)` | **bản nộp chuẩn** |
| 4. Sắp xếp + nhị phân | cặp (tổng, chỉ số) | `O(n log n)` | `O(n)` | không muốn băm |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `max_len_hash([1, 2], 3)`; (b) `max_len_hash([0, 0, 0], 0)`; (c) `max_len_hash([1, 2, 3], 7)`; (d) tổng tiền tố (có số `0` đầu) của `[2, -1, 3]`.

**Bài 2 (tính bằng tay).** Chạy Cách 3 cho `nums = [3, 1, -4, 2]`, `k = 0`: ghi `s`, `first`, `best` sau mỗi bước.

**Bài 3 (điền TODO).** Hoàn thiện `max_len_todo(nums, k)`.
#### Code demo

In [21]:
def max_len_todo(nums, k):
    # TODO 1: first = {0: -1}; s = 0; best = 0
    # TODO 2: với mỗi i, x: s += x; nếu s - k có trong first thì cập nhật best
    # TODO 3: chỉ ghi first[s] = i nếu s chưa có
    pass


# Tự làm xong rồi mới mở test:
# for (args, expected) in examples: assert max_len_todo(*args) == expected

**Bài 4 (tự viết lại).** Viết `longest_equal_01(nums)` trả độ dài mảng con liên tiếp dài nhất có số `0` bằng số `1` (LeetCode 525).

**Bài 5 (biến thể / mini challenge).** Viết `min_len_sum_k(nums, k)` trả độ dài **nhỏ nhất** của mảng con có tổng `k` (với số âm cho phép). Gợi ý: lưu chỉ số **cuối cùng** của mỗi tổng tiền tố. Thử `[1, -1, 5, -2, 3], 3` → `1` (đoạn `[3]`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: đoạn dài nhất có tổng `k`; Bài 2: `first = {0: -1}`; Bài 4: đổi `0 → -1`; Bài 5: ngắn nhất cần chỉ số **gần nhất**.
- **Mức 2 (vừa):** Bài 1: `2`, `3`, `0`, `[0, 2, 1, 4]`. Bài 2: `s = 3, 4, 0, 2`; `first = {0: -1, 3: 0, 4: 1}`; đoạn `[3, 1, -4]` dài `3`. Bài 3: khung Cách 3.
- **Mức 3 (sâu):** Bài 4: `max_len_hash([1 if x else -1 for x in nums], 0)`. Bài 5: duyệt `i`, tra `last[s - k]` (chỉ số gần nhất `< i`), ứng viên `i - last[s - k]`; **cập nhật** `last[s] = i` **sau** khi tra.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [22]:
# Bài 1
assert max_len_hash([1, 2], 3) == 2 and max_len_hash([0, 0, 0], 0) == 3 and max_len_hash([1, 2, 3], 7) == 0
assert [0] + list(accumulate([2, -1, 3])) == [0, 2, 1, 4]

# Bài 2
assert max_len_hash([3, 1, -4, 2], 0) == 3


# Bài 3
def max_len_todo_ans(nums, k):
    first = {0: -1}
    s = best = 0
    for i, x in enumerate(nums):
        s += x
        if s - k in first:
            best = max(best, i - first[s - k])
        if s not in first:
            first[s] = i
    return best


for (args, expected) in examples:
    assert max_len_todo_ans(*args) == expected


# Bài 4
def longest_equal_01(nums):
    return max_len_hash([1 if x else -1 for x in nums], 0)


assert longest_equal_01([0, 1]) == 2 and longest_equal_01([0, 1, 0]) == 2 and longest_equal_01([1, 1, 1]) == 0


# Bài 5
def min_len_sum_k(nums, k):
    last = {0: -1}
    s = 0
    best = 0
    for i, x in enumerate(nums):
        s += x
        if s - k in last:
            length = i - last[s - k]
            if best == 0 or length < best:
                best = length
        last[s] = i                                  # chỉ số GẦN NHẤT
    return best


def min_len_oracle(nums, k):
    best = 0
    for j in range(len(nums)):
        for i in range(j, len(nums)):
            if sum(nums[j:i + 1]) == k and (best == 0 or i - j + 1 < best):
                best = i - j + 1
    return best


assert min_len_sum_k([1, -1, 5, -2, 3], 3) == 1
rng = random.Random(17)
for _ in range(500):
    nums = [rng.randint(-3, 3) for _ in range(rng.randint(1, 10))]
    k = rng.randint(-5, 5)
    assert min_len_sum_k(nums, k) == min_len_oracle(nums, k), (nums, k)
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tổng tiền tố + bảng băm chỉ số đầu tiên)

```python
class Solution:
    def maxSubArrayLen(self, nums: List[int], k: int) -> int:
        first = {0: -1}
        s = best = 0
        for i, x in enumerate(nums):
            s += x
            if s - k in first:
                best = max(best, i - first[s - k])
            if s not in first:
                first[s] = i
        return best
```

**Checklist**

- [ ] Biểu diễn tổng đoạn bằng **hiệu hai tổng tiền tố**?
- [ ] Bảng băm lưu **chỉ số ĐẦU TIÊN** (không ghi đè)?
- [ ] Khởi tạo `{0: -1}` cho đoạn bắt đầu từ đầu mảng?
- [ ] **Tra cứu trước, ghi sau**?
- [ ] Không dùng hai con trỏ vì có **số âm**?
- [ ] Trả `0` khi không có đoạn nào?
- [ ] Đã thử `[0,0,0]` với `k = 0`, `[1,2]` với `k = 3`, `n = 2·10^5`?
- [ ] Skeleton đúng `class Solution` / `def maxSubArrayLen(self, nums: List[int], k: int) -> int`?